In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("../data")
amazon_books_data = pd.read_csv(f"{BASE_DIR}/amazon_books_reviews/books_data.csv")

### select column

In [2]:
amazon_books_data = amazon_books_data[["Title", "description", "authors", "publisher", "publishedDate", "categories"]]

### column published_year
- only keep year
- convert to numeric value

In [3]:
amazon_books_data['published_year'] = amazon_books_data['publishedDate'].str[:4]

In [4]:
amazon_books_data['published_year'] = pd.to_numeric(
    amazon_books_data['published_year'], errors='coerce'
).astype('Int64')

In [22]:
amazon_books_data['published_year'].unique()

<IntegerArray>
[1996, 2005, 2000, 2009, 1995, 1994, 2018, 2012, 2010, 2016,
 ...
 1783, 1765, 1826, 1751, 1793, 1711, 1778, 1695, 1791, 1732]
Length: 278, dtype: Int64

In [6]:
amazon_books_data.drop(columns=['publishedDate'], inplace=True)

In [23]:
amazon_books_data['published_year'].isna().sum()

np.int64(601)

### clean string

In [8]:
amazon_books_data['authors'] = amazon_books_data['authors'] \
    .str.strip("[]") \
    .str.replace("'", "", regex=False) \
    .str.replace('"', '', regex=False) \
    .str.strip()

amazon_books_data['categories'] = amazon_books_data['categories'] \
    .str.strip("[]") \
    .str.replace("'", "", regex=False) \
    .str.replace('"', '', regex=False) \
    .str.strip()

### rename columns

In [9]:
amazon_books_data = amazon_books_data.rename(columns={
    'Title': 'title',
    'authors': 'author',
    'publisher': 'publisher',
    'categories': 'genre'
})

### drop duplicate

In [10]:
amazon_books_data.duplicated(subset=['title', 'author']).sum()

np.int64(0)

### remove invalid and noisy values

In [11]:
# remove very long weird values
amazon_books_data = amazon_books_data[
    amazon_books_data['genre'].str.len() < 50
]

# remove null-like
amazon_books_data['genre'] = amazon_books_data['genre'].replace('None', None)

In [12]:
amazon_books_data = amazon_books_data.replace(['None', 'nan', '', 'NaN'], None)

In [13]:
amazon_books_data.head(20)

,title,description,author,publisher,genre,published_year
0,Its Only Art If Its Well Hung!,NaN,Julie Strain,NaN,Comics & Graphic Novels,1996
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,Philip Nel,A&C Black,Biography & Autobiography,2005
2,Wonderful Worship in Smaller Churches,This resource includes twelve principles in un...,David R. Ray,NaN,Religion,2000
3,Whispers of the Wicked Saints,Julia Thomas finds her life spinning out of co...,Veronica Haddon,iUniverse,Fiction,2005
5,The Church of Christ: A Biblical Ecclesiology ...,In The Church of Christ: A Biblical Ecclesiolo...,Everett Ferguson,Wm. B. Eerdmans Publishing,Religion,1996
8,Saint Hyacinth of Poland,The story for children 10 and up of St. Hyacin...,Mary Fabyan Windeatt,Tan Books & Pub,Biography & Autobiography,2009
9,Rising Sons and Daughters: Life Among Japan's ...,Wardell recalls his experience as a foreign st...,Steven Wardell,Plympton PressIntl,Social Science,1995
10,Muslim Women's Choices: Religious Belief and S...,Counters the Western views and stereotypes of ...,"Camillia Fawzi El-Solh, Judy Mabro",Berg Pub Limited,Religion,1994
11,Dramatica for Screenwriters,Dramatica for Screenwriters by Armando Saldana...,Armando Salda A-Mora,NaN,Reference,2005
12,Mensa Number Puzzles (Mensa Word Games for Kids),Acclaimed teacher and puzzler Evelyn B. Christ...,Evelyn B. Christensen,Sky Pony,Juvenile Nonfiction,2018


In [24]:
import csv
# save output
amazon_books_data.to_csv(BASE_DIR / "clean" / "amazon_books_data_clean.csv", index=False, na_rep='', quoting=csv.QUOTE_ALL, escapechar='\\')